# YOLO11 trainen op de BWB-dakendataset

Dit notebook draait op **Kaggle** met een GPU.

**Voorbereiding:**
1. Upload de dataset-zip met de Kaggle CLI (zie de README van de repo) of via *Datasets → New Dataset*.
2. Maak een nieuw notebook, klik rechts op **Add Input** en koppel je dataset.
3. Zet onder *Settings → Accelerator* een **GPU** aan (bijv. T4 x2 of P100).

De getrainde gewichten verschijnen na afloop onder *Output* als `best.pt`.

In [ ]:
%pip install -q ultralytics

In [ ]:
# Dataset opzoeken in /kaggle/input en klaarzetten in /kaggle/working.
# Kaggle pakt geüploade zips meestal automatisch uit; dit celletje vindt de
# data.yaml ongeacht de exacte mapstructuur.
import glob, pathlib, shutil, yaml

kandidaten = glob.glob('/kaggle/input/**/data.yaml', recursive=True)
assert kandidaten, 'Geen data.yaml gevonden — is de dataset als Input gekoppeld?'
bron = pathlib.Path(kandidaten[0]).parent
print('Dataset gevonden in:', bron)

werkmap = pathlib.Path('/kaggle/working/dataset')
if not werkmap.exists():
    shutil.copytree(bron, werkmap)

cfg = yaml.safe_load((werkmap / 'data.yaml').read_text())
cfg['path'] = str(werkmap)
(werkmap / 'data.yaml').write_text(yaml.safe_dump(cfg))
print('Klassen:', cfg['names'])

In [ ]:
from ultralytics import YOLO

# yolo11s is een goed startpunt; yolo11m/l geven meer kwaliteit maar trainen trager.
model = YOLO('yolo11s.pt')
resultaten = model.train(
    data=str(werkmap / 'data.yaml'),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project='/kaggle/working/runs',
    name='bwb_daken',
)

In [ ]:
# Validatie: mAP-scores per klasse.
metrics = model.val(data=str(werkmap / 'data.yaml'))
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

In [ ]:
# Visuele steekproef: voorspellingen op een paar validatietegels.
import random
from IPython.display import Image as IPyImage, display

val_beelden = list((werkmap / 'images' / 'val').glob('*.jpg')) + \
              list((werkmap / 'images' / 'val').glob('*.jpeg'))
steekproef = random.sample(val_beelden, k=min(4, len(val_beelden)))
voorspellingen = model.predict([str(p) for p in steekproef], save=True,
                               project='/kaggle/working/runs', name='voorspellingen')
for r in voorspellingen:
    display(IPyImage(filename=str(pathlib.Path(r.save_dir) / pathlib.Path(r.path).name)))

## Resultaat

- Beste gewichten: `/kaggle/working/runs/bwb_daken/weights/best.pt` — te downloaden via het *Output*-tabblad.
- Trainingscurves en confusion matrix staan in `/kaggle/working/runs/bwb_daken/`.

**Volgende stap:** gebruik `best.pt` lokaal om pre-annotaties te genereren voor de
nieuwe klassen (zonnepanelen, dakkapel, ...), corrigeer die in Label Studio en
train opnieuw met de uitgebreide dataset.

*Bevat gegevens van PDOK: Luchtfoto Beeldmateriaal Nederland (CC-BY 4.0) en de BAG.*